# Preprocess Datasets V2 (Logged, NaN-safe, Proper Encoding Before SMOTE/SMOTENC)

In [11]:
# Config
PM_CSV = "data/predictive_maintenance.csv"
EA_CSV = "data/equipment_anomaly_data.csv"
ME_CSV = "data/marine_engine_data.csv"

ENCODING = "onehot"           # "onehot" -> use OneHot + SMOTE; "label" -> Label-encode + SMOTENC
BALANCE = "auto"              # "none", "smote", "smotenc", "auto"
TARGET_MODE = "status"        # "status", "failure", "health"
TIMESTAMP_FREQ = "D"          # "D" or "M"
OUT_DIR = "./processed"
SEED = 42

import os, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE, SMOTENC

warnings.filterwarnings("ignore")
os.makedirs(OUT_DIR, exist_ok=True)
print("[INFO] Config loaded")

[INFO] Config loaded


In [12]:
# Column specs
PM_ID = ["UDI", "Product ID"]
PM_CAT = ["Type"]
PM_TGT1 = ["Target"]
PM_TGT2 = ["Failure Type"]
PM_NUM = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

EA_CAT = ["equipment", "location"]
EA_TGT = ["faulty"]
EA_NUM = ["temperature", "pressure", "vibration", "humidity"]

ME_TS  = ["timestamp"]
ME_ID  = ["engine_id"]
ME_CAT = ["failure_mode", "fuel_type", "engine_type", "manufacturer"]
ME_TGT = ["maintenance_status"]
ME_NUM = ["engine_temp", "oil_pressure", "fuel_consumption", "vibration_level", "rpm",
          "engine_load", "coolant_temp", "exhaust_temp", "running_period", "fuel_consumption_per_hour"]

RENAME_PM = {
    "Air temperature [K]": "air_temperature",
    "Process temperature [K]": "process_temperature",
    "Rotational speed [rpm]": "rpm",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear",
    "Type": "type",
    "Failure Type": "failure_type",
}
RENAME_EA = {
    "temperature": "air_temperature",
    "pressure": "pressure",
    "vibration": "vibration",
    "humidity": "humidity",
}

UNIFIED_NUM_CANDIDATES = [
    "air_temperature", "process_temperature", "rpm", "torque", "tool_wear", "pressure", "vibration", "humidity",
    "engine_temp", "oil_pressure", "fuel_consumption", "vibration_level", "engine_load",
    "coolant_temp", "exhaust_temp", "running_period", "fuel_consumption_per_hour"
]
UNIFIED_CAT_CANDIDATES = [
    "type", "failure_type", "equipment", "location", "failure_mode",
    "fuel_type", "engine_type", "manufacturer"
]

def to_datetime_safe(s): return pd.to_datetime(s, errors="coerce")

def detect_class_imbalance(y):
    vc = pd.Series(y).value_counts(normalize=True)
    return (len(vc) > 1) and (vc.min() < 0.3)

def map_status(series):
    return series.astype(str).map({"Normal":0,"Requires Maintenance":1,"Critical":2}).fillna(0).astype(int)

def infer_health_label_row(row):
    if "Target" in row and pd.notna(row["Target"]) and int(row["Target"]) == 1: return 1
    if "Failure Type" in row and isinstance(row["Failure Type"], str) and row["Failure Type"].lower() not in ["nan","none","no failure"]: return 1
    if "faulty" in row and pd.notna(row["faulty"]) and int(row["faulty"]) == 1: return 1
    if "maintenance_status" in row and isinstance(row["maintenance_status"], str) and row["maintenance_status"] in ["Requires Maintenance","Critical"]: return 1
    return 0

print("[INFO] Specs ready")

[INFO] Specs ready


In [13]:
# Load CSVs
def load_pm(path):
    df = pd.read_csv(path); df.rename(columns=RENAME_PM, inplace=True); df["source"]="predictive_maintenance"; return df
def load_ea(path):
    df = pd.read_csv(path); df.rename(columns=RENAME_EA, inplace=True); df["source"]="equipment_anomaly"; return df
def load_me(path):
    df = pd.read_csv(path)
    if "timestamp" not in df.columns: raise ValueError("marine_engine: 'timestamp' missing")
    df["timestamp"] = to_datetime_safe(df["timestamp"]); df["source"]="marine_engine"; return df

try:
    df_pm, df_ea, df_me = load_pm(PM_CSV), load_ea(EA_CSV), load_me(ME_CSV)
    print("[RESULT] Loaded datasets shapes:", df_pm.shape, df_ea.shape, df_me.shape)
except Exception as e:
    print("[ERROR] Loading CSVs failed:", e)
    raise

display(df_pm.head()); display(df_ea.head()); display(df_me.head())

[RESULT] Loaded datasets shapes: (10000, 11) (7672, 8) (5200, 18)


,UDI,Product ID,type,air_temperature,process_temperature,rpm,torque,tool_wear,Target,failure_type,source
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure,predictive_maintenance
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure,predictive_maintenance
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure,predictive_maintenance
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure,predictive_maintenance
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure,predictive_maintenance


,air_temperature,pressure,vibration,humidity,equipment,location,faulty,source
0,58.180180,25.029278,0.606516,45.694907,Turbine,Atlanta,0.0,equipment_anomaly
1,75.740712,22.954018,2.338095,41.867407,Compressor,Chicago,0.0,equipment_anomaly
2,71.358594,27.276830,1.389198,58.954409,Turbine,San Francisco,0.0,equipment_anomaly
3,71.616985,32.242921,1.770690,40.565138,Pump,Atlanta,0.0,equipment_anomaly
4,66.506832,45.197471,0.345398,43.253795,Pump,New York,0.0,equipment_anomaly


,timestamp,engine_id,engine_temp,oil_pressure,fuel_consumption,vibration_level,rpm,engine_load,coolant_temp,exhaust_temp,running_period,fuel_consumption_per_hour,maintenance_status,failure_mode,engine_type,fuel_type,manufacturer,source
0,2023-01-01,ENG_001,79.816406,7.049409,1000.000000,4.366612,1770.214578,42.472407,78.323108,450.0,49.741791,100.0,Critical,Oil Leakage,4-stroke High-Speed,Diesel,MAN B&W,marine_engine
1,2023-01-08,ENG_001,98.982068,8.000000,6308.623817,3.732792,1677.238238,77.042858,100.000000,450.0,94.351515,100.0,Requires Maintenance,Oil Leakage,2-stroke Low-Speed,Diesel,Mitsubishi,marine_engine
2,2023-01-15,ENG_001,83.918153,8.000000,6444.402260,4.061372,1487.472085,63.919637,78.178337,450.0,120.095804,100.0,Normal,No Failure,2-stroke Medium-Speed,Diesel,Caterpillar,marine_engine
3,2023-01-22,ENG_001,81.887081,7.601603,4439.946613,3.999554,1548.624692,55.919509,82.896344,450.0,122.321555,100.0,Requires Maintenance,Mechanical Wear,2-stroke Medium-Speed,Diesel,MAN B&W,marine_engine
4,2023-01-29,ENG_001,78.550429,6.233033,3146.234038,4.520559,1441.151499,29.361118,80.791150,450.0,111.978460,100.0,Normal,No Failure,4-stroke High-Speed,Diesel,Wärtsilä,marine_engine


In [14]:

# Encoder builders with robust OHE handling
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

def build_unified_encoders_union(dfs, cat_cols_union, encoding):
    art = {"encoding": encoding}
    if encoding == "label":
        art["label_encoders"] = {}
        for c in cat_cols_union:
            vals = []
            for d in dfs:
                if c in d.columns:
                    vals.append(d[c].astype(str))
            if len(vals):
                allv = pd.concat(vals, axis=0).fillna("NA")
            else:
                allv = pd.Series(["__na__"])
            le = LabelEncoder(); le.fit(allv)
            art["label_encoders"][c] = {"classes_": list(le.classes_)}
        print("[INFO] Built union LabelEncoders for", len(cat_cols_union), "categorical columns.")
        return art
    else:
        frames = []
        for d in dfs:
            tmp = {}
            n = len(d)
            for c in cat_cols_union:
                if c in d.columns:
                    tmp[c] = d[c].astype(str).fillna("__na__")
                else:
                    tmp[c] = pd.Series(["__na__"] * n, index=d.index)
            frames.append(pd.DataFrame(tmp, index=d.index))
        cat_full = pd.concat(frames, axis=0)

        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        ohe.fit(cat_full)

        art["ohe"] = ohe
        art["ohe_in"] = list(cat_full.columns)
        print("[INFO] Built union OneHotEncoder over", len(art["ohe_in"]), "categorical columns.")
        return art

def transform_with_unified_encoders(df, num_cols, cat_cols, enc_art, encoding):
    X_num = df[[c for c in num_cols if c in df.columns]].copy()

    if encoding == "label":
        X_cat_list = []
        for c in cat_cols:
            vals = df[c].astype(str) if c in df.columns else pd.Series(["NA"] * len(df), index=df.index)
            classes = enc_art["label_encoders"][c]["classes_"]
            cat_type = pd.api.types.CategoricalDtype(categories=classes)
            codes = vals.astype(cat_type).cat.codes.replace(-1, np.nan).fillna(len(classes)).astype(int)
            X_cat_list.append(pd.Series(codes, name=c, index=df.index))
        X_cat = pd.concat(X_cat_list, axis=1) if len(X_cat_list) else pd.DataFrame(index=df.index)

        scaler = StandardScaler()
        if len(X_num.columns):
            X_num = pd.DataFrame(scaler.fit_transform(X_num), columns=list(X_num.columns), index=df.index)
        X_all = pd.concat([X_num, X_cat], axis=1)
        return X_all

    else:
        ohe = enc_art["ohe"]; ohe_in = enc_art["ohe_in"]
        data = {}
        for c in ohe_in:
            if c in df.columns:
                data[c] = df[c].astype(str).fillna("__na__")
            else:
                data[c] = pd.Series(["__na__"] * len(df), index=df.index)
        X_cat_full = pd.DataFrame(data, index=df.index)[ohe_in]

        scaler = StandardScaler()
        if len(X_num.columns):
            X_num = pd.DataFrame(scaler.fit_transform(X_num), columns=list(X_num.columns), index=df.index)

        cat_arr = ohe.transform(X_cat_full) if len(ohe_in) else np.zeros((len(df), 0))
        num_cols_out = list(X_num.columns)
        cat_cols_out = ohe.get_feature_names_out(ohe_in).tolist() if len(ohe_in) else []
        cols = num_cols_out + cat_cols_out

        num_block = X_num.values if len(num_cols_out) else np.zeros((len(df), 0))
        X_all = np.concatenate([num_block, cat_arr], axis=1)
        X_all_df = pd.DataFrame(X_all, columns=cols, index=df.index)
        return X_all_df

print("[INFO] Encoder utilities ready")

[INFO] Encoder utilities ready


In [15]:
# Per-dataset preprocessing
def preprocess_individual(df, id_cols, cat_cols, num_cols, target_cols, dataset_name, encoding, balance, target_mode, out_dir, seed=42):
    try:
        id_cols = [c for c in id_cols if c in df.columns]
        cat_cols = [c for c in cat_cols if c in df.columns]
        num_cols = [c for c in num_cols if c in df.columns]
        df_loc = df.copy()

        if "maintenance_status" in df_loc.columns:
            df_loc["maintenance_status_num"] = map_status(df_loc["maintenance_status"].astype(str))

        if target_mode == "status":
            df_loc["unified_target"] = df_loc["maintenance_status_num"] if "maintenance_status_num" in df_loc.columns else df_loc.apply(infer_health_label_row, axis=1).astype(int)
        elif target_mode == "failure":
            df_loc["unified_target"] = df_loc.get("failure_mode", df_loc.get("failure_type", "None")).astype(str)
            t_le = LabelEncoder(); df_loc["unified_target"] = t_le.fit_transform(df_loc["unified_target"].astype(str))
        else:
            df_loc["unified_target"] = df_loc.apply(infer_health_label_row, axis=1).astype(int)

        feat_cols = [c for c in (id_cols + cat_cols + num_cols) if c in df_loc.columns]
        df_loc = df_loc.dropna(subset=["unified_target"])

        ds_dir = Path(out_dir) / "individual" / dataset_name
        ds_dir.mkdir(parents=True, exist_ok=True)
        raw_path = ds_dir / "processed_v2.csv"
        bal_path = ds_dir / "processed_balanced_v2.csv"
        meta_path = ds_dir / "encoders_scalers_v2.json"

        enc_union = build_unified_encoders_union([df_loc], cat_cols, encoding)
        Xdf = transform_with_unified_encoders(df_loc, num_cols, cat_cols, enc_union, encoding)
        y = df_loc["unified_target"].copy()

        pd.concat([Xdf, y.rename("unified_target")], axis=1).to_csv(raw_path, index=False)
        print(f"[RESULT] Saved {dataset_name} unbalanced to {raw_path}")

        # Optional balancing
        Xb, yb = Xdf.copy(), y.copy()
        if balance in ["smote", "smotenc", "auto"] and detect_class_imbalance(y):
            Xb = Xb.fillna(Xb.mean(numeric_only=True)).fillna(0)
            good = yb.notna()
            Xb, yb = Xb.loc[good], yb.loc[good].astype(int)

            if encoding == "label":
                # Label-encoded categoricals already numeric; use SMOTENC if any categorical columns exist
                cat_idx = [Xb.columns.get_loc(c) for c in cat_cols if c in Xb.columns]
                if (balance in ["auto", "smotenc"]) and len(cat_idx):
                    sm = SMOTENC(categorical_features=cat_idx, random_state=seed)
                    Xb, yb = sm.fit_resample(Xb, yb)
                else:
                    sm = SMOTE(random_state=seed)
                    Xb, yb = sm.fit_resample(Xb, yb)
            else:
                # One-hot space -> SMOTE
                sm = SMOTE(random_state=seed)
                Xb, yb = sm.fit_resample(Xb, yb)

            pd.concat([pd.DataFrame(Xb, columns=Xdf.columns), pd.Series(yb, name="unified_target")], axis=1).to_csv(bal_path, index=False)
            print(f"[RESULT] Saved {dataset_name} balanced to {bal_path}")
        else:
            print(f"[INFO] Skipped balancing for {dataset_name} (class distribution adequate or BALANCE='none').")

        with open(meta_path, "w") as f: json.dump({"encoding": encoding, "balance": balance}, f, indent=2)
        return {"processed": str(raw_path), "processed_balanced": str(bal_path), "meta": str(meta_path)}
    except Exception as e:
        print(f"[ERROR] preprocess_individual({dataset_name}) failed:", e)
        raise

print("[INFO] Per-dataset routine ready")

[INFO] Per-dataset routine ready


In [16]:
# Run per-dataset preprocessing
try:
    per_pm = preprocess_individual(df_pm, PM_ID, ["type"], ["air_temperature","process_temperature","rpm","torque","tool_wear","pressure","vibration","humidity"], ["Target","failure_type"], "predictive_maintenance", ENCODING, BALANCE, TARGET_MODE, OUT_DIR, SEED)
    per_ea = preprocess_individual(df_ea, [], ["equipment","location"], ["air_temperature","pressure","vibration","humidity"], ["faulty"], "equipment_anomaly", ENCODING, BALANCE, TARGET_MODE, OUT_DIR, SEED)
    per_me = preprocess_individual(df_me, ["engine_id"], ["failure_mode","fuel_type","engine_type","manufacturer"], ME_NUM, ["maintenance_status","failure_mode"], "marine_engine", ENCODING, BALANCE, TARGET_MODE, OUT_DIR, SEED)
    print("[RESULT] Per-dataset outputs:", json.dumps({"pm": per_pm, "ea": per_ea, "me": per_me}, indent=2))
except Exception as e:
    print("[ERROR] Per-dataset run failed:", e)
    raise

[INFO] Built union OneHotEncoder over 1 categorical columns.
[RESULT] Saved predictive_maintenance unbalanced to processed\individual\predictive_maintenance\processed_v2.csv
[RESULT] Saved predictive_maintenance balanced to processed\individual\predictive_maintenance\processed_balanced_v2.csv
[INFO] Built union OneHotEncoder over 2 categorical columns.
[RESULT] Saved equipment_anomaly unbalanced to processed\individual\equipment_anomaly\processed_v2.csv
[RESULT] Saved equipment_anomaly balanced to processed\individual\equipment_anomaly\processed_balanced_v2.csv
[INFO] Built union OneHotEncoder over 4 categorical columns.
[RESULT] Saved marine_engine unbalanced to processed\individual\marine_engine\processed_v2.csv
[INFO] Skipped balancing for marine_engine (class distribution adequate or BALANCE='none').
[RESULT] Per-dataset outputs: {
  "pm": {
    "processed": "processed\\individual\\predictive_maintenance\\processed_v2.csv",
    "processed_balanced": "processed\\individual\\predicti

In [17]:

# Combined preprocessing (timestamp backbone + resampling + dense labels + encoding before balancing)
def synthesize_timestamps(n, low, high, seed=42):
    rng = np.random.default_rng(seed); start = pd.Timestamp(low); end = pd.Timestamp(high)
    delta = (end - start).total_seconds()
    if delta <= 0: return pd.date_range(start, periods=n, freq="D")
    offs = rng.uniform(0, delta, size=n); ts = [start + pd.to_timedelta(x, unit="s") for x in offs]
    return pd.to_datetime(ts)

try:
    # Align timestamps
    ts_min, ts_max = df_me["timestamp"].min(), df_me["timestamp"].max()
    if "timestamp" not in df_pm.columns: df_pm["timestamp"] = synthesize_timestamps(len(df_pm), ts_min, ts_max, SEED)
    else: df_pm["timestamp"] = to_datetime_safe(df_pm["timestamp"])
    if "timestamp" not in df_ea.columns: df_ea["timestamp"] = synthesize_timestamps(len(df_ea), ts_min, ts_max, SEED+7)
    else: df_ea["timestamp"] = to_datetime_safe(df_ea["timestamp"])
    print("[INFO] Timestamp alignment done.")

    # IDs
    if "engine_id" not in df_me.columns: df_me["engine_id"] = "engine_0"
    df_pm["engine_id"] = df_pm.get("engine_id", pd.Series([f"pm_{i}" for i in range(len(df_pm))]))
    df_ea["engine_id"] = df_ea.get("engine_id", pd.Series([f"ea_{i}" for i in range(len(df_ea))]))

    # Targets
    for d in [df_pm, df_ea, df_me]: d["health_label"] = d.apply(infer_health_label_row, axis=1).astype(int)
    df_me["maintenance_status_num"] = map_status(df_me["maintenance_status"].astype(str)) if "maintenance_status" in df_me.columns else 0

    if TARGET_MODE=="status":
        df_pm["unified_target"] = df_pm["health_label"].astype(int); df_ea["unified_target"] = df_ea["health_label"].astype(int); df_me["unified_target"] = df_me["maintenance_status_num"].astype(int)
    elif TARGET_MODE=="failure":
        df_pm["unified_target"] = df_pm.get("failure_mode", df_pm.get("failure_type","None")).astype(str)
        df_ea["unified_target"] = "None"; df_me["unified_target"] = df_me.get("failure_mode","None").astype(str)
        le_t = LabelEncoder(); all_t = pd.concat([df_pm["unified_target"].astype(str), pd.Series(df_ea["unified_target"]).astype(str), df_me["unified_target"].astype(str)])
        le_t.fit(all_t); df_pm["unified_target"]=le_t.transform(df_pm["unified_target"].astype(str)); df_ea["unified_target"]=le_t.transform(pd.Series(df_ea["unified_target"]).astype(str)); df_me["unified_target"]=le_t.transform(df_me["unified_target"].astype(str))
    else:
        df_pm["unified_target"] = df_pm["health_label"].astype(int); df_ea["unified_target"] = df_ea["health_label"].astype(int); df_me["unified_target"] = df_me["health_label"].astype(int)

    def keep_cols(df):
        base=["timestamp","engine_id","unified_target"]
        cats=[c for c in UNIFIED_CAT_CANDIDATES if c in df.columns]
        nums=[c for c in UNIFIED_NUM_CANDIDATES if c in df.columns]
        return df[base+cats+nums].copy(), cats, nums

    pm_keep, pm_cats, pm_nums = keep_cols(df_pm)
    ea_keep, ea_cats, ea_nums = keep_cols(df_ea)
    me_keep, me_cats, me_nums = keep_cols(df_me)

    all_cat_union = sorted(list(set(pm_cats + ea_cats + me_cats)))
    all_num_union = sorted(list(set(pm_nums + ea_nums + me_nums)))
    print("[INFO] Union columns computed. Categorical:", len(all_cat_union), "Numeric:", len(all_num_union))

    # Resample raw union (no encoding)
    raw_all = pd.concat([pm_keep, ea_keep, me_keep], ignore_index=True).sort_values(["engine_id","timestamp"])
    raw_all["timestamp"] = to_datetime_safe(raw_all["timestamp"])

    frames = []
    freq = TIMESTAMP_FREQ.upper()
    for eng, g in raw_all.groupby("engine_id"):
        g = g.set_index("timestamp").sort_index()
        g_num = g[all_num_union].resample(freq).mean()
        g_cat = g[all_cat_union].resample(freq).last()
        g_tgt = g[["unified_target"]].resample(freq).last()
        g_res = pd.concat([g_num, g_cat, g_tgt], axis=1)
        g_res["engine_id"] = eng
        frames.append(g_res.reset_index())

    combined_resampled = pd.concat(frames, ignore_index=True).sort_values(["engine_id","timestamp"]).reset_index(drop=True)
    print("[INFO] Resample complete. Shape:", combined_resampled.shape)

    # Densify labels per engine (fix NaN y)
    combined_resampled["unified_target"] = (
        combined_resampled
        .groupby("engine_id", group_keys=False)["unified_target"]
        .apply(lambda s: s.ffill().bfill())
    )

    # Build X and y
    Xc = combined_resampled.drop(columns=["unified_target", "engine_id", "timestamp"]).copy()
    yc = combined_resampled["unified_target"].copy()

    # Impute X (numeric mean then 0) to remove NaNs
    Xc = Xc.fillna(Xc.mean(numeric_only=True)).fillna(0)

    # Align & cast y
    mask = yc.notna(); Xc = Xc.loc[mask].copy(); yc = yc.loc[mask].astype(int)

    # Encode categoricals BEFORE balancing
    cat_in_X = [c for c in all_cat_union if c in Xc.columns]
    if ENCODING == "label":
        Xc_enc = Xc.copy()
        for c in cat_in_X:
            le = LabelEncoder()
            Xc_enc[c] = le.fit_transform(Xc_enc[c].astype(str).fillna("NA"))
        # Determine categorical indices for SMOTENC
        cat_idx = [Xc_enc.columns.get_loc(c) for c in cat_in_X]
        # Balance
        if (BALANCE in ["auto","smotenc"]) and len(cat_idx) and detect_class_imbalance(yc):
            sm = SMOTENC(categorical_features=cat_idx, random_state=SEED)
            Xc_bal, yc_bal = sm.fit_resample(Xc_enc, yc)
        elif (BALANCE in ["auto","smote"]) and detect_class_imbalance(yc):
            sm = SMOTE(random_state=SEED)
            Xc_bal, yc_bal = sm.fit_resample(Xc_enc, yc)
        else:
            Xc_bal, yc_bal = Xc_enc, yc
    else:
        # OneHot encode with pandas.get_dummies to guarantee numeric input to SMOTE
        Xc_enc = pd.get_dummies(Xc, columns=cat_in_X, dummy_na=True)
        if (BALANCE in ["auto","smote"]) and detect_class_imbalance(yc):
            sm = SMOTE(random_state=SEED)
            Xc_bal, yc_bal = sm.fit_resample(Xc_enc, yc)
        else:
            Xc_bal, yc_bal = Xc_enc, yc

    # Save outputs
    comb_dir = Path(OUT_DIR) / "combined"
    comb_dir.mkdir(parents=True, exist_ok=True)
    combined_path = comb_dir / f"combined_{freq}_processed_v2.csv"
    combined_bal_path = comb_dir / f"combined_{freq}_processed_balanced_v2.csv"

    # For reference, save the resampled with original columns plus imputed X snapshot
    combined_resampled.assign(**Xc).to_csv(combined_path, index=False)
    pd.concat([pd.DataFrame(Xc_bal, columns=Xc_bal.columns), pd.Series(yc_bal, name="unified_target")], axis=1).to_csv(combined_bal_path, index=False)

    print("[RESULT] Saved combined unbalanced to", combined_path)
    print("[RESULT] Saved combined balanced to", combined_bal_path)
except Exception as e:
    print("[ERROR] Combined preprocessing failed:", e)
    raise

[INFO] Timestamp alignment done.
[INFO] Union columns computed. Categorical: 8 Numeric: 17
[INFO] Resample complete. Shape: (53772, 28)
[RESULT] Saved combined unbalanced to processed\combined\combined_D_processed_v2.csv
[RESULT] Saved combined balanced to processed\combined\combined_D_processed_balanced_v2.csv


In [18]:
# Quick previews
try:
    p_bal = Path(OUT_DIR, "combined", f"combined_{TIMESTAMP_FREQ.upper()}_processed_balanced_v2.csv")
    print("[INFO] Preview file exists?", p_bal.exists())
    if p_bal.exists():
        preview_df = pd.read_csv(p_bal)
        print("[RESULT] Preview head:")
        display(preview_df.head())
except Exception as e:
    print("[ERROR] Preview failed:", e)

[INFO] Preview file exists? True
[RESULT] Preview head:


,air_temperature,coolant_temp,engine_load,engine_temp,exhaust_temp,fuel_consumption,fuel_consumption_per_hour,humidity,oil_pressure,pressure,...,manufacturer_Rolls-Royce,manufacturer_Wärtsilä,manufacturer_Yanmar,manufacturer_nan,type_0,type_H,type_L,type_M,type_nan,unified_target
0,200.552657,78.323108,42.472407,79.816406,450.000000,1000.00000,100.000000,50.016574,7.049409,35.738048,...,False,False,False,False,True,False,False,False,False,2
1,200.552657,85.000245,49.794979,85.067979,449.829456,3937.67453,116.270048,50.016574,7.281496,35.738048,...,False,False,False,False,True,False,False,False,False,2
2,200.552657,85.000245,49.794979,85.067979,449.829456,3937.67453,116.270048,50.016574,7.281496,35.738048,...,False,False,False,False,True,False,False,False,False,2
3,200.552657,85.000245,49.794979,85.067979,449.829456,3937.67453,116.270048,50.016574,7.281496,35.738048,...,False,False,False,False,True,False,False,False,False,2
4,200.552657,85.000245,49.794979,85.067979,449.829456,3937.67453,116.270048,50.016574,7.281496,35.738048,...,False,False,False,False,True,False,False,False,False,2


In [19]:
def assess_dataframe(df, target=None, high_card_threshold=0.5):
    print("Data Assessment Report")

    n_rows, n_cols = df.shape
    print(f"\nShape of Dataset: {n_rows} Rows x {n_cols} Columns")

    print("\nColumn Information:")
    info_table = pd.DataFrame({
        "DataType": df.dtypes,
        "Non-Null Count": df.notnull().sum(),
        "Null Count": df.isnull().sum(),
        "Null %": (df.isnull().sum() / n_rows * 100).round(2),
        "Unique Values": df.nunique(dropna=True),
        "Unique %": (df.nunique(dropna=True) / n_rows * 100).round(2),
    })
    print(info_table)

    dup_count = df.duplicated().sum()
    dup_percent = round((dup_count / n_rows) * 100, 2) if n_rows else 0.0
    print(f"\nDuplicate Rows: {dup_count} ({dup_percent}%)")

    const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("\nConstant Columns:")
    print(const_cols if const_cols else "None")

    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    high_card_cols = []
    for c in cat_cols:
        ratio = (df[c].nunique(dropna=True) / n_rows) if n_rows else 0.0
        if ratio >= high_card_threshold:
            high_card_cols.append((c, round(ratio*100, 2)))
    print("\nHigh Cardinality Categorical Columns (>= {:.0f}% Unique):".format(high_card_threshold*100))
    if high_card_cols:
        for c, pct in high_card_cols:
            print(f"- {c}: {pct}% Unique")
    else:
        print("None")

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print("\nNumeric Columns Summary:")
        modes = {}
        if len(num_cols) > 0:
            mode_df = df[num_cols].mode(dropna=True)
            for c in num_cols:
                modes[c] = mode_df[c].iloc[0] if not mode_df.empty and c in mode_df else np.nan
        summary = pd.DataFrame({
            "Mean": df[num_cols].mean(numeric_only=True),
            "Median": df[num_cols].median(numeric_only=True),
            "Mode": pd.Series(modes),
            "Min": df[num_cols].min(numeric_only=True),
            "Q1": df[num_cols].quantile(0.25, numeric_only=True),
            "Q3": df[num_cols].quantile(0.75, numeric_only=True),
            "Max": df[num_cols].max(numeric_only=True),
            "Standard Deviation": df[num_cols].std(numeric_only=True),
            "Skewness": df[num_cols].skew(numeric_only=True),
            "Kurtosis": df[num_cols].kurt(numeric_only=True)
        }).round(4)
        print(summary)

        print("\nOutlier Counts (IQR method):")
        outlier_rows = []
        for c in num_cols:
            series = df[c].dropna()
            if series.empty:
                outlier_rows.append((c, 0, 0.0))
                continue
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            if iqr == 0:
                outlier_rows.append((c, 0, 0.0))
                continue
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = ((series < lower) | (series > upper)).sum()
            pct = round(count / len(series) * 100, 2)
            outlier_rows.append((c, int(count), pct))
        if outlier_rows:
            outlier_table = pd.DataFrame(outlier_rows, columns=["Column", "Outliers", "Outliers %"])
            print(outlier_table)

        print("\nPearson Correlation (Numeric):")
        pearson_corr = df[num_cols].corr(method="pearson")
        print(pearson_corr.round(3))

        print("\nSpearman Correlation (Numeric):")
        spearman_corr = df[num_cols].corr(method="spearman")
        print(spearman_corr.round(3))
    else:
        print("\nNo Numeric columns found.")

    print("\nColumns > 50% Missing Values:")
    high_missing = info_table[info_table["Null %"] > 50].sort_values("Null %", ascending=False)
    if not high_missing.empty:
        print(high_missing[["Null Count", "Null %"]])
    else:
        print("None")

    print("\nMemory Usage:")
    total_mem = df.memory_usage(deep=True).sum()
    print(f"Total: {total_mem} bytes ({round(total_mem/1024,2)} KB, {round(total_mem/(1024*1024),2)} MB)")
    memory_per_col = df.memory_usage(deep=True)
    memory_table = pd.DataFrame({"Bytes": memory_per_col}).sort_values("Bytes", ascending=False)
    print("\nPer Column Memory:")
    print(memory_table)

    if target is not None and target in df.columns:
        print(f"\nClass Balance for '{target}':")
        vc = df[target].value_counts(dropna=False)
        vp = df[target].value_counts(normalize=True, dropna=False).mul(100).round(2)
        bal = pd.DataFrame({"Count": vc, "Percent %": vp})
        print(bal)

    print("\nData Assessment Completed")

In [20]:
assess_dataframe(preview_df, target="unified_target")

Data Assessment Report

Shape of Dataset: 85032 Rows x 67 Columns

Column Information:
                DataType  Non-Null Count  Null Count  Null %  Unique Values  \
air_temperature  float64           85032           0     0.0           8949   
coolant_temp     float64           85032           0     0.0           9313   
engine_load      float64           85032           0     0.0           9555   
engine_temp      float64           85032           0     0.0           9555   
exhaust_temp     float64           85032           0     0.0            430   
...                  ...             ...         ...     ...            ...   
type_H              bool           85032           0     0.0              2   
type_L              bool           85032           0     0.0              2   
type_M              bool           85032           0     0.0              2   
type_nan            bool           85032           0     0.0              1   
unified_target     int64           85032    